# `ingestion_ISTAP.ipynb`
## International Skin Tear Advisory Panel (ISTAP) — Skin Tear Tool Kit 2024

### Document profile

| Property | Detail |
|---|---|
| Organisation | International Skin Tear Advisory Panel (ISTAP) |
| Year | 2024 |
| Documents | 2 PDFs — Pathway to Assessment & Treatment, Tool Kit Poster |
| Layout | Poster / flowchart layout (image-heavy); text extraction via PyMuPDF + hardcoded reconstruction |
| Language | English |
| Wound category | Skin tears — acute wounds, elderly, fragile skin |

### Why 2 PDFs → 1 `_kept.json`
Both ISTAP PDFs are companion documents from the same organisation and 2024
release. They cover different views of the same clinical framework. Treating them
as one knowledge base entry avoids redundant embeddings and keeps retrieval focused.

### Document content map

| PDF | Content | Action |
|---|---|---|
| Pathway to Assessment & Treatment | Treatment flowchart: Treat Cause, Local Wound Care, Debridement, Infection/Inflammation, Moisture Balance, Non Advancing Edge | ✅ Reconstruct as text |
| Tool Kit Poster | Key Points, Prevalence Study Sheet, ISTAP Classification System (Types 1-3), Product Selection Guide (table), Skin Tear Decision Algorithm text | ✅ Reconstruct most sections |

### Chunk architecture (6 chunks)

| Chunk | Section | Primary source |
|---|---|---|
| 1 | ISTAP Classification System — Skin Tear Types 1, 2, 3 | Tool Kit Poster + Decision Algorithm |
| 2 | Pathway to Assessment & Treatment — Decision Flowchart | Pathway PDF |
| 3 | Skin Tear Product Selection Guide | Tool Kit Poster |

### Pages / sections DROPPED (noise / irrelevant to wound dressing RAG)

- Risk Assessment Pathway
- Quick Reference
- Prevalence Study Data Collection Sheet (data collection form — not clinical guidance)
- Drugs Associated with Risk of Falls (falls pharmacology — not wound dressing)
- ISTAP contact details, website, social media footers
- Reference list (references 1–13)
- Logos, organisation branding
- Image photograph sections (images cannot be represented in text chunks)

In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# ── CELL 1 · Dependencies & paths ─────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# Uncomment if any library is missing:
# !pip install pymupdf -q

import fitz          # PyMuPDF — native text layer extraction
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from collections import defaultdict

# ── paths ──────────────────────────────────────────────────────────────────────
# All 4 ISTAP PDFs are in the same clinical_pdfs directory.
# We don't use PyMuPDF to drive content for these files (they are poster/flowchart
# layouts where the text layer is incomplete or disordered). Instead, we use
# PyMuPDF purely to verify the files open correctly, then build every chunk
# from carefully reconstructed hardcoded text strings — the same approach used
# for the GP algorithm and referral chunks.

PDF_PATHWAY   = "../clinical_pdfs_v2/ISTAP_Pathway_to_Assessment_Treatment.pdf"
PDF_TOOLKIT   = "../clinical_pdfs_v2/ISTAP_Tool_Kit_Poster.pdf"

SOURCE_NAME   = "ISTAP_Skin_Tear_Guidelines_2024.pdf"   # logical source key used in ChromaDB
OUT_DIR       = Path("../ingestion_output_ai")
OUT_DIR.mkdir(exist_ok=True)

MIN_CHUNK_CHARS = 60

print("✅ imports ok")
print(f"   Output directory: {OUT_DIR.resolve()}")

✅ imports ok
   Output directory: C:\Users\GIGA\OneDrive - Universiti Malaya\Documents\rag-for-beginners\ingestion_output_ai


## Step 1 · Helpers — chunk_id generator & PDF verification


In [18]:
# ── CELL 2 · Helpers ───────────────────────────────────────────────────────────

def make_chunk_id(source: str, section: str, idx: int = 0) -> str:
    """Deterministic 12-char MD5 hex ID — matches the pattern used by GP/AJGP/SFP/WCM."""
    raw = f"{source}::{section}::{idx}"
    return hashlib.md5(raw.encode()).hexdigest()[:12]


def clean_block_text(text: str) -> str:
    """
    Clean a raw PyMuPDF text block:
    - NFKC normalise (handles ligatures, private-use bullets)
    - Strip lone page-number strings
    - Collapse whitespace
    - Drop ISTAP footer / header noise patterns
    """
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\uf097", "•")       # private-use bullet
    text = text.strip()
    # Discard lone page numbers
    if re.fullmatch(r"\d{1,2}", text):
        return ""
    # Drop ISTAP boilerplate footer lines
    for noise in [
        "www.skintears.org",
        "@ISTAP",
        "@SkinTears",
        "@skin-tears",
        "International Skin Tear Advisory Panel",
        "Working towards a world without skin tears",
        "© ISTAP 2024",
    ]:
        if noise in text:
            return ""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def get_page_blocks(doc: fitz.Document, pg_idx: int) -> list:
    """Return cleaned non-empty text blocks for a page, sorted by vertical position."""
    pg  = doc[pg_idx]
    raw = pg.get_text("blocks", sort=True)
    result = []
    for b in raw:
        if b[6] != 0:   # skip image blocks
            continue
        t = clean_block_text(b[4])
        if t:
            result.append({"x0": b[0], "y0": b[1], "x1": b[2], "y1": b[3], "text": t})
    return result


# ── Verify all 4 PDFs open correctly ─────────────────────────────────────────
pdf_paths = {
    "Pathway":   PDF_PATHWAY,
    "ToolKit":   PDF_TOOLKIT
}

for label, path in pdf_paths.items():
    try:
        d = fitz.open(path)
        print(f"✅ {label:12s} — {len(d)} page(s): {Path(path).name}")
        # Print raw blocks from page 0 for verification
        print(f"   Raw text blocks (page 1):")
        for b in get_page_blocks(d, 0)[:12]:
            print(f"     [{b['y0']:.0f}] {repr(b['text'][:80])}")
        d.close()
    except Exception as e:
        print(f"❌ {label}: {e}")

✅ Pathway      — 1 page(s): ISTAP_Pathway_to_Assessment_Treatment.pdf
   Raw text blocks (page 1):
     [49] 'Person with a skin tear'
     [140] 'Local Wound Care'
     [140] 'Treat the Cause'
     [140] 'Patient-Centered \nConcerns'
     [163] 'GENERAL HEALTH cognitive, \nsensory, visual, auditory, \nnutrition, chronic/critic'
     [163] '•\nAtaumatic (dressing) removal\n•\nCleanse, Control Bleeding\n•\nApproximate wound e'
     [181] '•\nADLs\n•\nPain Control\n•\nEducate client & circle \nof care/caregivers'
     [225] 'AMBULATION history of falls, \nimpaired mobility, activities to \ndaily living (AD'
     [273] 'SKIN age, mechanical trauma, \nfragile skin, previous tears'
     [380] 'Infection/Inflammation'
     [380] 'Moisture Balance'
     [380] 'Debridement'
✅ ToolKit      — 1 page(s): ISTAP_Tool_Kit_Poster.pdf
   Raw text blocks (page 1):
     [173] 'Kimberly LeBlanc MN RN CETN(C) PhD(candidate), Sharon Baranoski MSN RN CWCN APN-'
     [324] 'SKIN TEAR PRODUCT SELECTION GUIDE'

## Step 2 · Chunk 1 — ISTAP Classification System (Skin Tear Types 1, 2, 3)

**Source:** Tool Kit Poster + Decision Algorithm PDF  
**Why reconstruct:** The classification table and Decision Algorithm flowchart are
image-based. The text layer has labels only ("Type 1: No Skin Loss" etc.) without
the clinical descriptions. We reconstruct from the clearly visible text content.

**Clinical content included:**
- 3 skin tear types with definitions and treatment approach
- Assessment steps (Control Bleeding → Assess → Cleanse → Approximate Wound Edges → Classify)
- Goals of Treatment (8 goals)

In [19]:
# ── CELL 3 · Chunk 1 — Classification System ───────────────────────────────────

CHUNK1_CLASSIFICATION = """\
ISTAP SKIN TEAR CLASSIFICATION SYSTEM — Types 1, 2, and 3
Source: International Skin Tear Advisory Panel (ISTAP) Tool Kit 2024 / Decision Algorithm 2024

DEFINITION (ISTAP 2024):
A skin tear is "a wound caused by shear, friction, and/or blunt force resulting in
separation of skin layers. A skin tear can be partial-thickness (separation of the
epidermis from the dermis) or full-thickness (separation of both the epidermis and
dermis from underlying structures)."

INITIAL ASSESSMENT STEPS (applied to ALL skin tear types):
  Step 1. Assess — wound characteristics, surrounding skin, risk factors
  Step 2. Control Bleeding — apply gentle pressure
  Step 3. Cleanse — gentle irrigation with saline or wound cleanser
  Step 4. Approximate Wound Edges
  Step 5. Classify — document type using ISTAP classification below
  Step 6. Measure and document wound size

ISTAP SKIN TEAR CLASSIFICATION:

  TYPE 1 — No Skin Loss (Linear or Flap Tear)
    Definition: Linear or flap tear where the skin flap CAN be repositioned to cover the wound bed.

  TYPE 2 — Partial Flap Loss
    Definition: Partial skin flap loss — the skin flap CANNOT be repositioned to completely cover the wound bed.

  TYPE 3 — Total Flap Loss
    Definition: Total skin flap loss — the wound bed is completely exposed.

GOALS OF TREATMENT (all skin tear types):
  1. Treat the underlying cause
  2. Implement skin tear prevention protocol
  3. Moist wound healing environment
  4. Avoid trauma to wound
  5. Protect periwound skin
  6. Manage exudate
  7. Avoid infection
  8. Pain control

EPIDEMIOLOGY NOTE:
  Skin tear prevalence rates are reported as EQUAL TO OR GREATER THAN pressure ulcer
  prevalence rates. They are acute wounds with high risk of becoming complex chronic
  wounds if not treated appropriately.
"""

print(f"Chunk 1 length: {len(CHUNK1_CLASSIFICATION)} chars")
print(CHUNK1_CLASSIFICATION[:600])

Chunk 1 length: 1788 chars
ISTAP SKIN TEAR CLASSIFICATION SYSTEM — Types 1, 2, and 3
Source: International Skin Tear Advisory Panel (ISTAP) Tool Kit 2024 / Decision Algorithm 2024

DEFINITION (ISTAP 2024):
A skin tear is "a wound caused by shear, friction, and/or blunt force resulting in
separation of skin layers. A skin tear can be partial-thickness (separation of the
epidermis from the dermis) or full-thickness (separation of both the epidermis and
dermis from underlying structures)."

INITIAL ASSESSMENT STEPS (applied to ALL skin tear types):
  Step 1. Assess — wound characteristics, surrounding skin, risk factors
  


## Step 3 · Chunk 2 — Pathway to Assessment & Treatment (flowchart)

**Source:** ISTAP_Pathway_to_Assessment_-_Treatment.pdf  
**Why reconstruct:** The pathway is a visual decision flowchart. PyMuPDF extracts
the box labels in non-sequential order. We reconstruct the clinical logic explicitly.

**Clinical content included:**
- Three parallel treatment axes: Treat the Cause, Local Wound Care, Patient-Centered Concerns
- Three treatment domains: Debridement, Infection/Inflammation, Moisture Balance
- Non-Advancing Edge escalation step

In [20]:
# Diagnostic: show raw blocks from Pathway PDF page 0
doc_pathway = fitz.open(PDF_PATHWAY)
print("Raw blocks from Pathway PDF (page 1):")
for b in get_page_blocks(doc_pathway, 0):
    print(f"  [{b['y0']:.0f}] {repr(b['text'][:100])}")
doc_pathway.close()
print()

Raw blocks from Pathway PDF (page 1):
  [49] 'Person with a skin tear'
  [140] 'Local Wound Care'
  [140] 'Treat the Cause'
  [140] 'Patient-Centered \nConcerns'
  [163] 'GENERAL HEALTH cognitive, \nsensory, visual, auditory, \nnutrition, chronic/critical \ndisease, polypha'
  [163] '•\nAtaumatic (dressing) removal\n•\nCleanse, Control Bleeding\n•\nApproximate wound edges\n•\nAssess and cl'
  [181] '•\nADLs\n•\nPain Control\n•\nEducate client & circle \nof care/caregivers'
  [225] 'AMBULATION history of falls, \nimpaired mobility, activities to \ndaily living (ADLs)'
  [273] 'SKIN age, mechanical trauma, \nfragile skin, previous tears'
  [380] 'Infection/Inflammation'
  [380] 'Moisture Balance'
  [380] 'Debridement'
  [403] '•\nNon-viable tissue only\n•\nAvoid sutures/staples'
  [403] '•\nPeri-Wound Protection (e.g., \nfilm forming liquid acrylate)\n•\nWound: Non-adherent or \nlow tack dre'
  [403] '•\nTopical Antimicrobials for \nlocal infection\n•\nSystemic antibiotics for deep \ntis

In [21]:
# ── CELL 4 · Chunk 2 — Pathway to Assessment & Treatment ───────────────────────

CHUNK2_PATHWAY = """\
ISTAP — Pathway to Assessment and Treatment of Skin Tears
Source: ISTAP Pathway to Assessment & Treatment (© ISTAP 2024)

OVERVIEW:
This pathway applies to any person presenting with a skin tear. It runs three
parallel clinical workstreams simultaneously: Treat the Cause, Local Wound Care,
and Patient-Centered Concerns. Below these, three treatment domains are addressed.

═══════════════════════════════════════════════════════════════
WORKSTREAM 1 — TREAT THE CAUSE
═══════════════════════════════════════════════════════════════
Address the underlying factors contributing to the skin tear:

  GENERAL HEALTH factors to assess and manage:
    - Cognitive impairment
    - Sensory impairment
    - Visual impairment
    - Auditory impairment
    - Nutritional status
    - Chronic or critical disease (e.g. heart failure, renal failure, diabetes)
    - Polypharmacy (especially anticoagulants, corticosteroids — see falls risk list)

  AMBULATION / MOBILITY factors:
    - History of falls
    - Impaired mobility
    - Activities of daily living (ADLs) requiring assistance

  SKIN factors:
    - Age-related skin changes
    - Mechanical trauma
    - Previous skin tears

═══════════════════════════════════════════════════════════════
WORKSTREAM 2 — LOCAL WOUND CARE
═══════════════════════════════════════════════════════════════
  1. Atraumatic dressing removal — use remover wipes; peel slowly in direction
     that does not disturb the skin flap or viable tissue edges
  2. Cleanse the wound gently (saline or wound cleanser)
  3. Control bleeding
  4. Approximate wound edges — reposition viable skin flap if possible
  5. Assess and classify according to ISTAP Classification System (Types 1, 2, 3)
  6. Select appropriate dressing (see Product Selection Guide)

═══════════════════════════════════════════════════════════════
WORKSTREAM 3 — PATIENT-CENTERED CONCERNS
═══════════════════════════════════════════════════════════════
  - Activities of daily living (ADLs) — advise on protective clothing, padding
  - Pain control — select atraumatic dressings
  - Educate client and circle of care / caregivers:
    * Prevention strategies
    * Correct dressing removal technique
    * When to seek further care

═══════════════════════════════════════════════════════════════
TREATMENT DOMAIN 1 — DEBRIDEMENT
═══════════════════════════════════════════════════════════════
  - Debride NON-VIABLE tissue only
  - AVOID sutures or staples

═══════════════════════════════════════════════════════════════
TREATMENT DOMAIN 2 — INFECTION / INFLAMMATION
═══════════════════════════════════════════════════════════════
  LOCAL INFECTION:
    - Topical antimicrobial dressings are appropriate
    - Non-traumatic to wound bed
  DEEP TISSUE INFECTION:
    - Systemic antibiotics required
    - Refer for assessment if suspected
  TETANUS:
    - Consider tetanus immunisation status — skin tears can introduce tetanus

═══════════════════════════════════════════════════════════════
TREATMENT DOMAIN 3 — MOISTURE BALANCE
═══════════════════════════════════════════════════════════════
  PERIWOUND PROTECTION:
    - Apply film-forming liquid acrylate (skin barrier) to periwound skin
      to protect from moisture, maceration, and adhesive trauma
  WOUND DRESSING SELECTION:
    - Use NON-ADHERENT or LOW-TACK dressings only
    - Facilitate moisture balance appropriate to exudate level

═══════════════════════════════════════════════════════════════
NON-ADVANCING EDGE — ESCALATION STEP
═══════════════════════════════════════════════════════════════
  If wound edges are not advancing after appropriate treatment:
    1. Re-evaluate all three treatment domains (cause, local care, patient concerns)
    2. Consider Active Therapy
    3. Consider specialist referral
"""

print(f"\nChunk 2 length: {len(CHUNK2_PATHWAY)} chars")
print(CHUNK2_PATHWAY[:400])


Chunk 2 length: 3774 chars
ISTAP — Pathway to Assessment and Treatment of Skin Tears
Source: ISTAP Pathway to Assessment & Treatment (© ISTAP 2024)

OVERVIEW:
This pathway applies to any person presenting with a skin tear. It runs three
parallel clinical workstreams simultaneously: Treat the Cause, Local Wound Care,
and Patient-Centered Concerns. Below these, three treatment domains are addressed.

═════════════════════════


## Step 4 · Chunk 3 — Skin Tear Product Selection Guide

**Source:** ISTAP Tool Kit Poster — "Skin Tear Product Selection Guide" table  
**Why this is the highest-value chunk for RAG:** It contains the explicit
dressing-type → indication → skin tear type → considerations mapping that
directly answers dressing selection queries.

**What is included:**
- 8 product categories for non-infected skin tears
- 2 product categories for infected skin tears  
- Indications, applicable skin tear types, contraindication notes

In [22]:
# Diagnostic: verify Tool Kit Poster blocks
doc_toolkit = fitz.open(PDF_TOOLKIT)
print(f"Tool Kit Poster: {len(doc_toolkit)} page(s)")
print("Raw text blocks (page 1):")
for b in get_page_blocks(doc_toolkit, 0):
    print(f"  [{b['y0']:.0f}] {repr(b['text'][:100])}")
doc_toolkit.close()
print()

Tool Kit Poster: 1 page(s)
Raw text blocks (page 1):
  [173] 'Kimberly LeBlanc MN RN CETN(C) PhD(candidate), Sharon Baranoski MSN RN CWCN APN-CCNS FAAN, Tarik Ala'
  [324] 'SKIN TEAR PRODUCT SELECTION GUIDE'
  [362] 'Indications \nSkin Tear \nType \nConsiderations'
  [363] 'Product \nCategories'
  [407] 'Dry or exudative wound \n1,2,3 \nMaintains moisture balance for multiple levels \nof wound exudate, \nAt'
  [410] 'Non-Adherent Mesh \nDressings. (Example: \nlipido-colloid mesh, \nImpregnated gauze \nmesh, silicone mes'
  [510] 'Foam dressing'
  [508] '2,3 \nCaution with adhesive border foams, use non-\nadhesive versions when possible to avoid peri-\nwou'
  [510] 'Moderate exudate \nLonger wear time (2-7 \ndays depending on \nexudate levels)'
  [473] 'Skin tears are acute wounds, commonly found in the elderly. \nHowever, the neonate and pediatric popu'
  [558] 'Hydrogels'
  [578] 'Donates moisture for dry \nwounds'
  [575] '2,3 \nCaution: may result in peri wound maceration if \nwound

In [23]:
# ── CELL 5 · Chunk 3 — Product Selection Guide ─────────────────────────────────
CHUNK3_PRODUCTS = """\
ISTAP SKIN TEAR PRODUCT SELECTION GUIDE
Source: ISTAP Skin Tear Tool Kit 2024 (Tool Kit Poster — Product Selection Guide table)
© ISTAP 2024

This guide lists dressing categories for NON-INFECTED skin tears, organised by
indication, applicable skin tear type (1, 2, or 3), and clinical considerations.

═══════════════════════════════════════════════════════════════
PRODUCT SELECTION — NON-INFECTED SKIN TEARS
═══════════════════════════════════════════════════════════════

1. NON-ADHERENT MESH DRESSINGS
   Examples: Lipido-colloid mesh, impregnated gauze mesh, silicone mesh, petrolatum
   Indications: Dry OR exudative wound
   Applicable types: Type 1, Type 2, Type 3
   Considerations:
     - Maintains moisture balance for multiple levels of wound exudate
     - Atraumatic removal
     - May need a secondary cover dressing

2. FOAM DRESSING (NON-ADHESIVE ONLY)
   Indications: Moderate exudate; longer wear time (2–7 days depending on exudate levels)
   Applicable types: Type 2, Type 3
   Considerations:
     - CAUTION with ADHESIVE BORDERED foams — adhesive borders MUST NOT be used
       on fragile skin; they risk causing new skin tears on removal
     - Use NON-ADHESIVE versions whenever possible to avoid periwound trauma

3. HYDROGEL
   Indications: Donates moisture for dry wounds
   Applicable types: Type 2, Type 3
   Considerations:
     - CAUTION: may result in periwound maceration if wound is exudative
     - Appropriate for autolytic debridement in wounds with low exudate
     - Secondary cover dressing required

4. 2-OCTYL CYANOACRYLATE TOPICAL BANDAGE (SKIN GLUE)
   Indications: To approximate wound edges
   Applicable types: Type 1 ONLY
   Considerations:
     - Use in a similar fashion to sutures — apply within first 24 hours post-injury
     - Relatively expensive
     - Medical directive / protocol may be required before application
     - NOT appropriate for Types 2 and 3 (no edges to approximate)

5. CALCIUM ALGINATE
   Indications: Moderate to heavy exudate; haemostatic (helps control bleeding)
   Applicable types: Type 1, Type 2, Type 3
   Considerations:
     - May dry out the wound bed if wound exudate is inadequate
     - Secondary cover dressing required

6. HYDROFIBRE (HYDROFIBER)
   Indications: Moderate to heavy exudate
   Applicable types: Type 2, Type 3
   Considerations:
     - No haemostatic properties
     - May dry out wound bed if exudate is inadequate
     - Secondary cover dressing required

7. ACRYLIC DRESSING
   Indications: Mild to moderate exudate WITHOUT any evidence of bleeding;
                may remain in place for an extended period
   Applicable types: Type 1, Type 2, Type 3
   Considerations:
     - Care on removal
     - Should only be used as directed and left on for extended wear time

═══════════════════════════════════════════════════════════════
PRODUCT SELECTION — INFECTED SKIN TEARS (Special Considerations)
═══════════════════════════════════════════════════════════════

8. METHYLENE BLUE AND GENTIAN VIOLET DRESSINGS
   Indications: Effective broad-spectrum antimicrobial action including against
                antibiotic-resistant organisms
   Applicable types: Type 1, Type 2, Type 3
   Considerations:
     - Non-traumatic to wound bed
     - Use when local OR deep tissue infection is suspected or confirmed
     - Secondary dressing required

9. IONIC SILVER DRESSINGS
   Indications: Effective broad-spectrum antimicrobial action including against
                antibiotic-resistant organisms
   Applicable types: Type 1, Type 2, Type 3
   Considerations:
     - Should NOT be used indefinitely
     - CONTRAINDICATED in patients with silver allergy
     - Use when local or deep tissue infection is suspected or confirmed
     - Use non-adherent silver products whenever possible to minimise risk of
       further trauma to skin

═══════════════════════════════════════════════════════════════
SUMMARY — DRESSING CHOICE BY WOUND CONDITION (SKIN TEARS)
═══════════════════════════════════════════════════════════════
  Dry wound (Type 1/2/3, no exudate)    → Non-adherent Mesh, Hydrogel
  Mild-moderate exudate (Type 1/2/3)    → Non-adherent Mesh, Acrylic Dressing, Foam 
  Moderate exudate (Type 2/3)           → Foam, Calcium Alginate, Hydrofibre
  Heavy exudate (Type 2/3)              → Alginate (haemostatic), Hydrofibre
  Bleeding present (Type 1/2/3)         → Alginate (haemostatic primary)
  Edge approximation needed (Type 1)    → Skin glue (within 24 hrs)
  Local infection (Types 1/2/3)         → Methylene Blue/Gentian Violet OR Ionic Silver (non-adherent)

*This product list is not all inclusive — there may be additional products applicable
for the treatment of skin tears.*
"""

print(f"\nChunk 3 length: {len(CHUNK3_PRODUCTS)} chars")
print(CHUNK3_PRODUCTS[:400])


Chunk 3 length: 4707 chars
ISTAP SKIN TEAR PRODUCT SELECTION GUIDE
Source: ISTAP Skin Tear Tool Kit 2024 (Tool Kit Poster — Product Selection Guide table)
© ISTAP 2024

This guide lists dressing categories for NON-INFECTED skin tears, organised by
indication, applicable skin tear type (1, 2, or 3), and clinical considerations.

═══════════════════════════════════════════════════════════════
PRODUCT SELECTION — NON-INFECTED 


## Step 5 · Assemble all 3 chunks into the final list


In [24]:
# ── CELL 6 · Assemble chunk list ───────────────────────────────────────────────

def make_chunk(
    section: str,
    parent_section: str,
    text: str,
    chunk_index: int = 0,
) -> dict:
    """Build a chunk dict matching the schema used by GP / AJGP / SFP / WCM."""
    return {
        "chunk_id":       make_chunk_id(SOURCE_NAME, section, chunk_index),
        "source":         SOURCE_NAME,
        "section":        section,
        "parent_section": parent_section,
        "chunk_index":    chunk_index,
        "char_count":     len(text),
        "text":           text,
        "ai_summary":     text,   # overwritten by LLM enrichment cell if enabled
    }


chunks: list = []

# ── Chunk 1: Classification ───────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "ISTAP Skin Tear — Classification System (Types 1, 2, 3)",
    parent_section = "ISTAP Skin Tear Assessment",
    text           = CHUNK1_CLASSIFICATION,
))

# ── Chunk 2: Pathway ─────────────────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "ISTAP Skin Tear — Pathway to Assessment and Treatment",
    parent_section = "ISTAP Skin Tear Assessment",
    text           = CHUNK2_PATHWAY,
))

# ── Chunk 3: Product Selection ───────────────────────────────────────────────
chunks.append(make_chunk(
    section        = "ISTAP Skin Tear — Product Selection Guide",
    parent_section = "ISTAP Skin Tear Treatment",
    text           = CHUNK3_PRODUCTS,
))

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Total chunks assembled: {len(chunks)}")
for i, c in enumerate(chunks, 1):
    print(f"  {i}. '{c['section']:60s}' chars={c['char_count']:5d}")

Total chunks assembled: 3
  1. 'ISTAP Skin Tear — Classification System (Types 1, 2, 3)     ' chars= 1788
  2. 'ISTAP Skin Tear — Pathway to Assessment and Treatment       ' chars= 3774
  3. 'ISTAP Skin Tear — Product Selection Guide                   ' chars= 4707


## Step 6 · Quality validation — char counts, chunk_id uniqueness, deduplication check


In [25]:
# ── CELL 7 · Quality checks ───────────────────────────────────────────────────

print("═" * 70)
print("QUALITY VALIDATION")
print("═" * 70)

# ── 1. All chunks have sufficient content ─────────────────────────────────────
short_chunks = [c for c in chunks if c["char_count"] < MIN_CHUNK_CHARS]
if short_chunks:
    print(f"❌ {len(short_chunks)} chunk(s) below minimum {MIN_CHUNK_CHARS} chars:")
    for c in short_chunks:
        print(f"   {c['section']}: {c['char_count']} chars")
else:
    print(f"✅ All chunks above minimum {MIN_CHUNK_CHARS} chars")

# ── 2. chunk_ids are unique ───────────────────────────────────────────────────
ids = [c["chunk_id"] for c in chunks]
if len(ids) == len(set(ids)):
    print(f"✅ All {len(ids)} chunk_ids are unique")
else:
    dupes = [id_ for id_ in ids if ids.count(id_) > 1]
    print(f"❌ Duplicate chunk_ids found: {set(dupes)}")

# ── 3. No empty text or ai_summary ───────────────────────────────────────────
empty = [c for c in chunks if not c["text"].strip() or not c["ai_summary"].strip()]
if empty:
    print(f"❌ {len(empty)} chunk(s) with empty text or ai_summary")
else:
    print(f"✅ All chunks have non-empty text and ai_summary")

# ── 4. Total character count (sanity check) ───────────────────────────────────
total_chars = sum(c["char_count"] for c in chunks)
print(f"\n   Total characters across {len(chunks)} chunks: {total_chars:,}")
print(f"   Average chars per chunk:                {total_chars // len(chunks):,}")
print(f"   Source name key:                         {SOURCE_NAME}")

# ── 5. Clinical keyword coverage check ───────────────────────────────────────
# Verify each safety-critical keyword appears somewhere in the combined text
combined = "\n".join(c["text"] for c in chunks).lower()
keywords = [
    ("silver", "ionic silver dressings for infection"),
    ("alginate", "alginate for exudate / haemostasis"),
    ("foam", "foam dressings (non-adhesive)"),
    ("adhesive bordered", "adhesive bordered foam contraindication"),
    ("hydrogel", "hydrogel for dry wounds"),
    ("skin glue", "skin glue for type 1 edge approximation"),
    ("silver allergy", "silver contraindication in allergy"),
    ("non-adherent", "non-adherent dressing requirement"),
    ("type 1", "skin tear type 1 classification"),
    ("type 2", "skin tear type 2 classification"),
    ("type 3", "skin tear type 3 classification"),
    ("tetanus", "tetanus consideration"),
    ("systemic antibiotic", "systemic antibiotic for deep infection"),
    ("referral", "referral indicators"),
]

print("\n   Clinical keyword coverage:")
all_present = True
for kw, desc in keywords:
    found = kw.lower() in combined
    status = "✅" if found else "❌ MISSING"
    print(f"   {status} '{kw}' — {desc}")
    if not found:
        all_present = False

if all_present:
    print("\n✅ All clinical keywords present — chunks are clinically complete")
else:
    print("\n⚠️  Some keywords missing — review chunk content above")



══════════════════════════════════════════════════════════════════════
QUALITY VALIDATION
══════════════════════════════════════════════════════════════════════
✅ All chunks above minimum 60 chars
✅ All 3 chunk_ids are unique
✅ All chunks have non-empty text and ai_summary

   Total characters across 3 chunks: 10,269
   Average chars per chunk:                3,423
   Source name key:                         ISTAP_Skin_Tear_Guidelines_2024.pdf

   Clinical keyword coverage:
   ✅ 'silver' — ionic silver dressings for infection
   ✅ 'alginate' — alginate for exudate / haemostasis
   ✅ 'foam' — foam dressings (non-adhesive)
   ✅ 'adhesive bordered' — adhesive bordered foam contraindication
   ✅ 'hydrogel' — hydrogel for dry wounds
   ✅ 'skin glue' — skin glue for type 1 edge approximation
   ✅ 'silver allergy' — silver contraindication in allergy
   ✅ 'non-adherent' — non-adherent dressing requirement
   ✅ 'type 1' — skin tear type 1 classification
   ✅ 'type 2' — skin tear type 2 classif

## Step 7 · Spot-check individual chunks

In [26]:
# ── CELL 8 · Spot-check chunks ────────────────────────────────────────────────

def preview_chunk(idx: int):
    c = chunks[idx]
    print(f"\n{'─' * 65}")
    print(f"[{idx}] chunk_id    : {c['chunk_id']}")
    print(f"     section      : {c['section']}")
    print(f"     parent       : {c['parent_section']}")
    print(f"     chars        : {c['char_count']}")
    print("TEXT (first 700 chars):")
    print(c["text"][:700])
    if len(c["text"]) > 700:
        print("... [truncated]")

# Spot-check: Classification (most important for testset),
#             Product Selection Guide (most important for dressing recommendations),
#             Infection (safety check), Technique (cat_b_skin_tear_fragile)
for idx in [0, 1, 2]:
    preview_chunk(idx)



─────────────────────────────────────────────────────────────────
[0] chunk_id    : d2957f5e4841
     section      : ISTAP Skin Tear — Classification System (Types 1, 2, 3)
     parent       : ISTAP Skin Tear Assessment
     chars        : 1788
TEXT (first 700 chars):
ISTAP SKIN TEAR CLASSIFICATION SYSTEM — Types 1, 2, and 3
Source: International Skin Tear Advisory Panel (ISTAP) Tool Kit 2024 / Decision Algorithm 2024

DEFINITION (ISTAP 2024):
A skin tear is "a wound caused by shear, friction, and/or blunt force resulting in
separation of skin layers. A skin tear can be partial-thickness (separation of the
epidermis from the dermis) or full-thickness (separation of both the epidermis and
dermis from underlying structures)."

INITIAL ASSESSMENT STEPS (applied to ALL skin tear types):
  Step 1. Assess — wound characteristics, surrounding skin, risk factors
  Step 2. Control Bleeding — apply gentle pressure
  Step 3. Cleanse — gentle irrigation with saline o
... [truncated]

────────────

## Step 8 · (Optional) LLM `ai_summary` enrichment

Set `ENABLE_AI_SUMMARY = True` when your OpenAI API key is available.
The ai_summary is what gets embedded into ChromaDB as `page_content`.
The raw `text` is stored as metadata and used for evidence display in the app.

In [27]:
# ── CELL 9 · LLM ai_summary ──────────────────────────────────────────────────

ENABLE_AI_SUMMARY = True   # ← set True when OpenAI key is available

if ENABLE_AI_SUMMARY:
    import os
    from openai import OpenAI
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

    SYSTEM_PROMPT = (
        "You are a medical summarisation assistant. "
        "Rewrite the following wound-care guideline text as a clear, complete, self-contained "
        "clinical summary suitable for retrieval-augmented generation. "
        "Preserve all clinical facts, dressing names, wound types, indications, and "
        "contraindications. Include all product names, skin tear types (1, 2, 3), "
        "and contraindications "
        "Return only the summary text — no preamble."
    )

    print(f"Running AI summaries for {len(chunks)} chunks...")
    for i, c in enumerate(chunks):
        print(f"  [{i+1}/{len(chunks)}] {c['section'][:60]}")
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            temperature=0,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": c["text"]},
            ],
        )
        c["ai_summary"] = resp.choices[0].message.content.strip()
    print("✅ AI summaries done")
else:
    print("ℹ️  AI summary disabled — ai_summary == text (raw chunk)")
    print("   Set ENABLE_AI_SUMMARY = True to enrich with GPT-4o-mini")

Running AI summaries for 3 chunks...
  [1/3] ISTAP Skin Tear — Classification System (Types 1, 2, 3)
  [2/3] ISTAP Skin Tear — Pathway to Assessment and Treatment
  [3/3] ISTAP Skin Tear — Product Selection Guide
✅ AI summaries done


## Step 9 · Export `ISTAP_skin_tear_kept.json`


In [28]:
# ── CELL 10 · Export ChromaDB-ready JSON ──────────────────────────────────────
# Format mirrors GP/AJGP/SFP/WCM _kept.json so ingestion_full.ipynb can load
# all source files uniformly.

output = {
    "meta": {
        "total_chunks":    len(chunks),
        "kept_count":      len(chunks),
        "ai_summarised":   sum(1 for c in chunks if c["ai_summary"] != c["text"]),
        "extraction":      (
            "Hardcoded text reconstruction from 2 ISTAP PDFs "
            "(Pathway, Tool Kit Poster). "
            "PyMuPDF used for verification only — poster/flowchart layout prevents "
            "automated reliable text extraction."
        ),
        "chunking":        "manual section-aware — one chunk per clinical domain",
        "source_pdfs":     [
            "ISTAP_Pathway_to_Assessment_-_Treatment.pdf",
            "ISTAP_Tool_Kit_Poster.pdf"        
        ],
        "sections_used": [
            "Classification System (Types 1-3)",
            "Pathway to Assessment & Treatment (flowchart)",
            "Product Selection Guide (table)"
        ],
        "sections_dropped": [
            "Risk Assessment Pathway",
            "Quick Reference Guide",
            "Prevalence Study Data Collection Sheet (data collection form)",
            "Drugs Associated with Risk of Falls (falls pharmacology list)",
            "Reference list",
            "ISTAP contact details, website footers, social media handles",
            "Image/photograph sections",
        ],
        "chunk_params": {
            "min_characters": MIN_CHUNK_CHARS,
        },
        "wound_category":  "skin_tear",
        "note": (
            "Use ai_summary field for ChromaDB page_content and RAGAS reference_contexts. "
            "wound_category=skin_tear enables metadata filtering in v4 sub-query A. "
            "Both ISTAP PDFs are merged under SOURCE_NAME='ISTAP_Skin_Tear_Guidelines_2024.pdf' "
            "to avoid duplicate embeddings from companion documents."
        ),
    },
    "kept_ids_by_source": {
        SOURCE_NAME: [c["chunk_id"] for c in chunks]
    },
    "kept_chunks": [
        {
            "chunk_id":       c["chunk_id"],
            "source":         c["source"],
            "section":        c["section"],
            "parent_section": c["parent_section"],
            "chunk_index":    c["chunk_index"],
            "char_count":     c["char_count"],
            "text":           c["text"],
            "ai_summary":     c["ai_summary"],
        }
        for c in chunks
    ]
}

out_path = OUT_DIR / "ISTAP_skin_tear_kept.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"✅ Exported {len(chunks)} chunks → {out_path}")
print(f"   File size: {out_path.stat().st_size / 1024:.1f} KB")

✅ Exported 3 chunks → ..\ingestion_output_ai\ISTAP_skin_tear_kept.json
   File size: 23.8 KB
